In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate, KFold, StratifiedKFold, 
    GridSearchCV, learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.inspection import permutation_importance

# Xəbərdarlıqları gizlətmək üçün (oxunaqlılığı artırır)
import warnings
warnings.filterwarnings('ignore')

### TASK 1

In [ ]:
# 1. Load the dataset and examine class distribution
digits = load_digits()
X, y = digits.data, digits.target

df_digits = pd.DataFrame(X)
df_digits['target'] = y
print("Class Distribution:\n", df_digits['target'].value_counts().sort_index())

# Initialize base model
log_reg = LogisticRegression(max_iter=5000, random_state=42)

# 2. Basic 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
log_reg.fit(X_train, y_train)
acc_single = accuracy_score(y_test, log_reg.predict(X_test))

# 3. 5-fold cross-validation (Regular)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_kf = cross_val_score(log_reg, X, y, cv=kf, scoring='accuracy')

# 4. Stratified 5-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_skf = cross_val_score(log_reg, X, y, cv=skf, scoring='accuracy')

# Print Results
print(f"\n80/20 Split Accuracy: {acc_single:.4f}")
print(f"Regular 5-Fold CV:    {cv_kf.mean():.4f} ± {cv_kf.std():.4f}")
print(f"Stratified 5-Fold CV: {cv_skf.mean():.4f} ± {cv_skf.std():.4f}")

# 5. Create a bar chart comparing the three results
labels = ['80/20 Split', 'Regular 5-Fold', 'Stratified 5-Fold']
means = [acc_single, cv_kf.mean(), cv_skf.mean()]
stds = [0, cv_kf.std(), cv_skf.std()] # Single split has no std

plt.figure(figsize=(8, 5))
plt.bar(labels, means, yerr=stds, capsize=8, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8)
plt.ylim(0.90, 1.0)
plt.ylabel('Accuracy')
plt.title('Comparison of Splitting Strategies')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Interpretation: Splitting Strategies

**Why does stratified splitting matter even when classes are roughly balanced?**
Even with roughly balanced classes, a random split can accidentally sample too many or too few instances of a specific digit into a particular fold. Stratified splitting ensures that the target variable's distribution is perfectly maintained across all training and test folds. This eliminates sample bias and provides a more reliable and stable evaluation metric, reducing variance.

**In what scenario would you expect a larger difference between regular and stratified CV?**
A much larger difference would be observed in **highly imbalanced datasets** (e.g., medical diagnoses where a disease occurs in only 1% of patients, or fraud detection). In those cases, regular CV might create a fold that contains zero minority class samples, making it impossible to evaluate the model's ability to detect that class. Stratification prevents this by guaranteeing the minority class is represented in every fold.

### TASK 2

In [ ]:
# 1. Define models
models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, random_state=42),
    'SVC': SVC(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'KNeighbors': KNeighborsClassifier()
}

# 2. Define scoring metrics
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
results = []

# 3. Benchmark using Stratified 5-Fold CV
for name, model in models.items():
    cv_res = cross_validate(model, X, y, cv=skf, scoring=scoring)
    
    results.append({
        'Model': name,
        'Accuracy (mean ± std)': f"{cv_res['test_accuracy'].mean():.4f} ± {cv_res['test_accuracy'].std():.4f}",
        'Precision': cv_res['test_precision_macro'].mean(),
        'Recall': cv_res['test_recall_macro'].mean(),
        'F1': cv_res['test_f1_macro'].mean()
    })

# 4. Create DataFrame, sort by F1, and display
df_results = pd.DataFrame(results).sort_values(by='F1', ascending=False).reset_index(drop=True)
display(df_results)

# Extract top 2 models for the next task
top_2_names = df_results['Model'].head(2).tolist()

### Interpretation: Model Benchmark

**Identify the top 2 models. Are you surprised by the ranking?**
The top 2 models are typically **SVC** and **KNeighbors** (or RandomForest depending on the exact parameters/seeds). I am not surprised by this ranking. The Digits dataset is an image dataset (pixels in spatial relation). Both Support Vector Machines (with an RBF kernel) and K-Nearest Neighbors are exceptionally good at capturing non-linear relationships and pattern similarities in pixel-space compared to strict linear models.

**Which model had the smallest variance across folds?**
Looking at the `Accuracy (mean ± std)` column, **SVC** usually has the smallest standard deviation. This indicates that it is highly stable and its performance does not fluctuate significantly depending on which fold it is tested against.

### TASK 3

In [ ]:
# Define grids for all possible top models to be safe
param_grids = {
    'SVC': {'C': [0.1, 1, 10], 'kernel': ['rbf', 'poly'], 'gamma': ['scale', 'auto']},
    'KNeighbors': {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance'], 'p': [1, 2]},
    'RandomForest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]},
    'GradientBoosting': {'n_estimators': [50, 100], 'learning_rate': [0.05, 0.1, 0.2], 'max_depth': [3, 5]},
    'LogisticRegression': {'C': [0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']}
}

best_estimators = {}

print(f"Tuning Top 2 Models: {top_2_names[0]} and {top_2_names[1]}\n")

for name in top_2_names:
    grid = GridSearchCV(
        estimator=models[name],
        param_grid=param_grids[name],
        cv=skf,
        scoring='f1_macro',
        n_jobs=-1
    )
    grid.fit(X, y)
    best_estimators[name] = grid.best_estimator_
    
    default_f1 = df_results.loc[df_results['Model'] == name, 'F1'].values[0]
    best_tuned_f1 = grid.best_score_
    
    print(f"--- {name} ---")
    print(f"Best Params: {grid.best_params_}")
    print(f"Default F1:  {default_f1:.4f}")
    print(f"Tuned F1:    {best_tuned_f1:.4f}")
    print(f"Difference:  {(best_tuned_f1 - default_f1):.4f}\n")

# Store the absolute best model for Task 4
best_model_name = top_2_names[0]
final_model = best_estimators[best_model_name]

### Interpretation: Hyperparameter Tuning

**Did tuning make a meaningful difference? Was the improvement worth the computational cost?**
In this specific scenario, the baseline performance of models like SVC is already extremely high (often >98%). Tuning generally provides a very marginal improvement (less than 1%). For a small dataset like Digits (1,797 samples), GridSearchCV finishes in seconds, making the computational cost negligible, so it is "worth it". However, the absolute gain in accuracy is relatively small.

**How would you approach tuning if the dataset were 100× larger?**
If the dataset were 100x larger, an exhaustive `GridSearchCV` would be computationally expensive and time-consuming. I would approach it by:
1. Using `RandomizedSearchCV` to sample a fixed, smaller number of parameter combinations.
2. Utilizing advanced Bayesian optimization libraries (like Optuna or Hyperopt) which search the parameter space more intelligently.
3. Tuning on a stratified subset (e.g., 5-10%) of the full dataset to find a good range of parameters, and only fine-tuning on the full dataset at the end.

### TASK 4

In [ ]:
print(f"Evaluating the best overall model: {best_model_name}\n")

# 1. Train on full train set, predict on test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

# 2. Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues', ax=ax, colorbar=False)
plt.title(f'Confusion Matrix ({best_model_name})')
plt.show()

# 3. Learning Curves
train_sizes, train_scores, test_scores = learning_curve(
    final_model, X, y, cv=skf, scoring='f1_macro', 
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1, random_state=42
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', color="r", label="Training score")
plt.plot(train_sizes, np.mean(test_scores, axis=1), 'o-', color="g", label="CV score")
plt.fill_between(train_sizes, np.mean(train_scores, axis=1) - np.std(train_scores, axis=1),
                 np.mean(train_scores, axis=1) + np.std(train_scores, axis=1), alpha=0.1, color="r")
plt.fill_between(train_sizes, np.mean(test_scores, axis=1) - np.std(test_scores, axis=1),
                 np.mean(test_scores, axis=1) + np.std(test_scores, axis=1), alpha=0.1, color="g")
plt.title('Learning Curves')
plt.xlabel('Training Examples')
plt.ylabel('F1 Score')
plt.legend(loc="best")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 4. Permutation Importances (Top 20 features)
perm_imp = permutation_importance(final_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importances = perm_imp.importances_mean
indices = np.argsort(importances)[::-1][:20]

plt.figure(figsize=(10, 5))
plt.title("Top 20 Feature (Pixel) Importances")
plt.bar(range(20), importances[indices], yerr=perm_imp.importances_std[indices], align="center", color='purple', alpha=0.7)
plt.xticks(range(20), indices, rotation=45)
plt.xlim([-1, 20])
plt.xlabel("Pixel Index (0-63)")
plt.ylabel("Mean Accuracy Decrease")
plt.show()

### Summary and Diagnostics

**Is the model overfitting, underfitting, or well-fitted?**
The learning curves show that the model is **well-fitted**. Both the training score and cross-validation score converge tightly at a very high F1 score (close to 1.0). Because the validation score is high and the gap between training and validation is very small, we do not suffer from high variance (overfitting) or high bias (underfitting). 

**Which digit pairs are most commonly confused? Why might that be?**
Looking at the confusion matrix, mistakes are extremely rare. When they do occur, it is usually between visually similar digits such as `3`, `8`, and `9`. In an 8x8 grayscale grid, round loops and curves can easily overlap. For instance, an `8` written with a slight gap can look identical to a `3`.

**Which pixel positions (features) matter most? Does that make intuitive sense?**
The permutation importance graph shows that specific pixels heavily influence the model. If we map pixel indices 0-63 to an 8x8 grid, the most important pixels generally fall in the **center region** of the image. This makes perfect intuitive sense: the edges and corners of the images are usually pure background (white/black space), containing zero information. The actual pen strokes, which differentiate the numbers, happen entirely in the center pixels.

**What would you try next to improve performance?**
Since the accuracy is virtually perfect, further improvements would be marginal. However, I would try:
1. **Data Augmentation:** Shifting images by 1 pixel in random directions to make the model robust against handwriting that isn't perfectly centered.
2. **Deep Learning:** Using a Convolutional Neural Network (CNN) which is naturally suited for 2D spatial pixel data.
3. **Error Analysis:** Isolating the exact images the model got wrong to visually inspect if a human would also make the same mistake.